In [5]:
import json
import pandas as pd

# Read JSON file
#with open("/home/lbjga/projects/omnifold/daily_14.json", "r", encoding="utf-8") as f:
#    data = json.load(f)

In [6]:
file_path = "/home/lbjga/projects/omnifold/daily_14.json"

# Read JSONL file into DataFrame
df = pd.read_json(file_path, lines=True)

# Display DataFrame
display(df)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

,city,time,data
0,"{'id': 1283240, 'name': 'Kathmandu', 'country'...",1489465075,"[{'dt': 1489471200, 'temp': {'day': 286.84, 'm..."
1,"{'id': 3632308, 'name': 'Merida', 'country': '...",1489465075,"[{'dt': 1489420800, 'temp': {'day': 284.63, 'm..."
2,"{'id': 1280737, 'name': 'Lhasa', 'country': 'C...",1489465075,"[{'dt': 1489471200, 'temp': {'day': 273.27, 'm..."
3,"{'id': 745042, 'name': 'İstanbul', 'country': ...",1489465075,"[{'dt': 1489399200, 'temp': {'day': 280.63, 'm..."
4,"{'id': 3496831, 'name': 'Mao', 'country': 'DO'...",1489465075,"[{'dt': 1489420800, 'temp': {'day': 296.15, 'm..."
...,...,...,...
22630,"{'id': 8199378, 'name': 'Modiin Ilit', 'countr...",1489466267,"[{'dt': 1489482000, 'temp': {'day': 290.65, 'm..."
22631,"{'id': 8199394, 'name': 'Ariel', 'country': 'I...",1489466267,"[{'dt': 1489482000, 'temp': {'day': 290.65, 'm..."
22632,"{'id': 8260318, 'name': 'Shanjeev Home', 'coun...",1489466267,"[{'dt': 1489471200, 'temp': {'day': 301.07, 'm..."
22633,"{'id': 8224933, 'name': 'PAJOK Pogee,', 'count...",1489466267,"[{'dt': 1489482000, 'temp': {'day': 307.83, 'm..."


Shape: (22635, 3)
Columns: ['city', 'time', 'data']


In [7]:
# Rename columns for easier access
df.columns = ["city_info", "timestamp", "forecast"]

display(df.head())

,city_info,timestamp,forecast
0,"{'id': 1283240, 'name': 'Kathmandu', 'country'...",1489465075,"[{'dt': 1489471200, 'temp': {'day': 286.84, 'm..."
1,"{'id': 3632308, 'name': 'Merida', 'country': '...",1489465075,"[{'dt': 1489420800, 'temp': {'day': 284.63, 'm..."
2,"{'id': 1280737, 'name': 'Lhasa', 'country': 'C...",1489465075,"[{'dt': 1489471200, 'temp': {'day': 273.27, 'm..."
3,"{'id': 745042, 'name': 'İstanbul', 'country': ...",1489465075,"[{'dt': 1489399200, 'temp': {'day': 280.63, 'm..."
4,"{'id': 3496831, 'name': 'Mao', 'country': 'DO'...",1489465075,"[{'dt': 1489420800, 'temp': {'day': 296.15, 'm..."


In [13]:
display(df['city_info'].head())

0    {'id': 1283240, 'name': 'Kathmandu', 'country'...
1    {'id': 3632308, 'name': 'Merida', 'country': '...
2    {'id': 1280737, 'name': 'Lhasa', 'country': 'C...
3    {'id': 745042, 'name': 'İstanbul', 'country': ...
4    {'id': 3496831, 'name': 'Mao', 'country': 'DO'...
Name: city_info, dtype: object

In [14]:
for item in df["city_info"].head():
    print(item)

{'id': 1283240, 'name': 'Kathmandu', 'country': 'NP', 'coord': {'lon': 85.316666, 'lat': 27.716667}}
{'id': 3632308, 'name': 'Merida', 'country': 'VE', 'coord': {'lon': -71.144997, 'lat': 8.598333}}
{'id': 1280737, 'name': 'Lhasa', 'country': 'CN', 'coord': {'lon': 91.099998, 'lat': 29.65}}
{'id': 745042, 'name': 'İstanbul', 'country': 'TR', 'coord': {'lon': 28.983311, 'lat': 41.03508}}
{'id': 3496831, 'name': 'Mao', 'country': 'DO', 'coord': {'lon': -71.078133, 'lat': 19.551861}}


In [ ]:
#1. How many city records are in the file?
num_cities = len(df)

print("Number of city records:", num_cities)

Number of city records: 22635


In [ ]:
#2. How many unique countries are represented?
df["country"] = df["city_info"].apply(lambda x: x["country"])

num_countries = df["country"].nunique()

print("Number of unique countries:", num_countries)

Number of unique countries: 243


In [ ]:
#2.1 inspect
countries = df["country"].unique()
print(countries)

<StringArray>
['NP', 'VE', 'CN', 'TR', 'DO', 'RU', 'PT', 'PL', 'IN', 'AU',
 ...
 'MP', 'MS', 'SB', 'SH', 'SJ', 'SM', 'SX', 'TL', 'TO', 'VI']
Length: 243, dtype: str


In [ ]:
#3. Which 10 countries have the most cities?
## value_counts() automatically counts the occurrences of each unique country and sorts them in descending order.
top_countries = (
    df["country"]
    .value_counts()
    .head(10)
    .reset_index()
)

top_countries.columns = ["country", "num_cities"]

display(top_countries)

,country,num_cities
0,US,2924
1,IN,2393
2,BR,1194
3,DE,985
4,RU,953
5,CN,788
6,JP,752
7,GB,672
8,IT,670
9,FR,610


In [17]:
#4. Explode each city’s daily forecast array into one row per (city, country, date). How many total rows?
# Explode forecast array
forecast_df = df.explode("forecast").copy()

# Extract city information
forecast_df["city"] = forecast_df["city_info"].apply(
    lambda x: x["name"]
)

forecast_df["country"] = forecast_df["city_info"].apply(
    lambda x: x["country"]
)

# Extract forecast date
forecast_df["date"] = pd.to_datetime(
    forecast_df["forecast"].apply(lambda x: x["dt"]),
    unit="s"
).dt.strftime("%Y-%m-%d")

print("Total forecast rows:", len(forecast_df))

display(forecast_df[["city", "country", "date"]].head())

Total forecast rows: 368909


,city,country,date
0,Kathmandu,NP,2017-03-14
0,Kathmandu,NP,2017-03-15
0,Kathmandu,NP,2017-03-16
0,Kathmandu,NP,2017-03-17
0,Kathmandu,NP,2017-03-18


In [ ]:
#5. Top 10 hottest cities based on average daytime temperature
#The temp.day value is in Kelvin.

#First extract the daytime temperature from each forecast.
forecast_df["temp_day"] = forecast_df["forecast"].apply(
    lambda x: x["temp"]["day"]
)

# Calculate mean daytime temperature for each city
city_temps = (
    forecast_df
    .groupby(["city", "country"])["temp_day"]
    .mean()
    .reset_index()
)

# Convert Kelvin to Celsius
city_temps["temp_day_C"] = city_temps["temp_day"] - 273.15

# Top 10 hottest cities
top_10_hottest = (
    city_temps
    .sort_values("temp_day", ascending=False)
    .head(10)
)

display(top_10_hottest[["city", "country", "temp_day_C"]])



,city,country,temp_day_C
5397,Dourbali,TD,39.083529
11569,Maiduguri,NG,38.745294
11531,Magumeri,NG,38.745294
4877,Damaturu,NG,38.739412
12034,Massakory,TD,38.686471
1379,Ayorou,NE,38.677647
15489,Potiskum,NG,38.615882
2716,Bogo,CM,38.589412
6644,Geidam,NG,38.577647
21461,Yagoua,CM,38.565882


In [19]:
#6. Build a tidy table for San Francisco, California.
#Include columns: date, temp_min, temp_max, humidity, weather_description.
#Convert temperatures from Kelvin to °C.
#Format the dates to be human readable, e.g.: YYYY‑MM‑DD

# Find San Francisco
sf_df = forecast_df[
    (forecast_df["city"] == "San Francisco") &
    (forecast_df["country"] == "US")
].copy()

# Extract forecast information
sf_df["temp_min"] = sf_df["forecast"].apply(
    lambda x: x["temp"]["min"] - 273.15
)

sf_df["temp_max"] = sf_df["forecast"].apply(
    lambda x: x["temp"]["max"] - 273.15
)

sf_df["humidity"] = sf_df["forecast"].apply(
    lambda x: x["humidity"]
)

sf_df["weather_description"] = sf_df["forecast"].apply(
    lambda x: x["weather"][0]["description"]
)

# Build tidy table
sf_tidy = sf_df[
    ["date", "temp_min", "temp_max",
     "humidity", "weather_description"]
].copy()

# Round temperatures
sf_tidy[["temp_min", "temp_max"]] = (
    sf_tidy[["temp_min", "temp_max"]].round(2)
)

# Sort by date
sf_tidy = sf_tidy.sort_values("date").reset_index(drop=True)

display(sf_tidy)

,date,temp_min,temp_max,humidity,weather_description
0,2017-03-13,9.81,18.95,67,sky is clear
1,2017-03-14,7.45,25.93,55,sky is clear
2,2017-03-15,8.63,22.75,56,sky is clear
3,2017-03-16,10.18,18.75,0,light rain
4,2017-03-17,6.50,18.56,0,light rain
5,2017-03-18,9.20,16.79,0,light rain
6,2017-03-19,10.50,18.08,0,light rain
7,2017-03-20,6.82,18.35,0,light rain
8,2017-03-21,4.57,16.67,0,sky is clear
9,2017-03-22,3.85,16.50,0,light rain
